# Week 2 EDA: Data Quality and Classification Risk

This notebook examines the licensed Breast Cancer Wisconsin (Diagnostic) subset used by the repository. It focuses on missingness, class imbalance, obvious leakage indicators, feature distributions, and error-sensitive evaluation. The analysis is educational and is not a clinical validation.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split

from src.data_quality import analyze_data_quality, format_report

DATA_PATH = Path('../data/breast_cancer_wisconsin_diagnostic.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/breast_cancer_wisconsin_diagnostic.csv')
data = pd.read_csv(DATA_PATH)
print(f'Shape: {data.shape}')
print(f'Columns: {list(data.columns)}')

Shape: (569, 3)
Columns: ['feature_a', 'feature_b', 'label']


In [2]:
quality = analyze_data_quality(DATA_PATH)
print(format_report(quality))

Rows: 569
Missing values: 0
Class counts: 0=212, 1=357
Majority/minority ratio: 1.684 (moderate)
Non-numeric features: none
Target-copy features: none
Feature/target correlations: feature_a=-0.730, feature_b=-0.415
High target-correlation features: none
Duplicate feature rows: 0
Conflicting duplicate groups: 0


In [3]:
data.describe().round(3)

       feature_a  feature_b    label
count    569.000    569.000  569.000
mean      14.127     19.290    0.627
std        3.524      4.301    0.484
min        6.981      9.710    0.000
25%       11.700     16.170    0.000
50%       13.370     18.840    1.000
75%       15.780     21.800    1.000
max       28.110     39.280    1.000

## Distribution findings

The dataset has no missing values. Benign observations are the majority (62.7%), giving a 1.684 majority/minority ratio. Tukey's 1.5×IQR rule identifies 14 mean-radius and 7 mean-texture observations as potential outliers; they are retained because they are plausible measurements and may carry diagnostic signal. Neither feature directly copies the label, but mean radius has a substantial association with it. Correlation alone is not proof of leakage.

In [4]:
features = data[['feature_a', 'feature_b']]
labels = data['label']
x_train, x_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, random_state=42, stratify=labels
)

rows = []
for name, class_weight in [('original', None), ('class-balanced', 'balanced')]:
    predictions = LogisticRegression(
        random_state=42, class_weight=class_weight
    ).fit(x_train, y_train).predict(x_test)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, predictions),
        'benign_f1': f1_score(y_test, predictions),
        'malignant_recall': recall_score(y_test, predictions, pos_label=0),
        'false_negatives': fp,
        'false_positives': fn,
    })
pd.DataFrame(rows).set_index('model').round(3)

                 accuracy  benign_f1  malignant_recall  false_negatives  false_positives
model                                                                                     
original             0.888      0.912             0.830                9                7
class-balanced       0.888      0.908             0.906                5               11

## Decision

Accuracy and benign-class F1 hid the weaker malignant recall of 0.830. Class balancing raises malignant recall to 0.906 and reduces malignant cases predicted as benign from 9 to 5 while preserving accuracy at 0.888. The repository therefore adopts class-balanced Logistic Regression and promotes malignant recall to a regression-tested signal. The trade-off is four additional benign cases predicted as malignant, so this remains a learning baseline rather than a clinical decision rule.